## 🔍 기업마당 금융 탭 크롤링 

In [ ]:
import os
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import re
from urllib.parse import urlparse, parse_qs
import requests
import zipfile


import os
from dotenv import load_dotenv
import pickle
import faiss
import numpy as np
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
import re
import os
from langchain.document_loaders import PDFPlumberLoader, TextLoader
from langchain_community.vectorstores import FAISS

from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnableMap
from langchain.callbacks import get_openai_callback



# 셀레니움 옵션 설정
options = Options()
options.add_argument("--headless")
options.add_argument("--disable-gpu")
driver = webdriver.Chrome(options=options)

In [29]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")  # 환경변수 설정

In [ ]:
# 대상 URL
base_url = (
    "https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C128/AS/74/list.do"
    "?hashCode=01&rowsSel=6&cat=&article_seq=&pblancId=&schJrsdCodeTy="
    "&schWntyAt=&schAreaDetailCodes=&schEndAt=N&orderGb=&sort="
    "&condition=searchPblancNm&condition1=AND&preKeywords=&keyword=&rows=15"
)
detail_base = "https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C128/AS/74/"

In [ ]:
# 📌 마지막 페이지 수 추출 함수
def get_last_page_num():
    driver.get(base_url + "&cpage=1")
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    
    last_page_tag = soup.select_one("div.page_wrap a[title='마지막페이지']")
    if last_page_tag:
        href = last_page_tag.get("href")
        match = re.search(r"cpage=(\d+)", href)
        if match:
            return int(match.group(1))
    return 1  # 기본값

# 마지막 페이지 수 구하기
max_pages = get_last_page_num()
print(f"📄 총 페이지 수: {max_pages}")

📄 총 페이지 수: 17


In [ ]:
# 저장할 데이터
data = []

# 목록 페이지 파싱 함수
def parse_current_page_html(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = soup.select("div.table_Type_1 > table > tbody > tr")
    
    for row in rows:
        try:
            title_tag = row.select_one("td.txt_l a")
            title = title_tag.get_text(strip=True)
            href = title_tag.get("href")
            full_link = detail_base + href if href else None

            item = {
                "제목": title,
                "링크": full_link
            }
            data.append(item)
        except Exception as e:
            print("⚠️ 파싱 오류:", e)

# 페이지 수 설정 (원하는 만큼 수집)
# max_pages = get_last_page_num() -----------------> 얘 써도 됨
max_pages = 17
print(f"▶ 총 수집 페이지 수: {max_pages}")

for page in range(1, max_pages + 1):
    driver.get(f"{base_url}&cpage={page}")
    time.sleep(2)
    parse_current_page_html(driver.page_source)
    print(f"✅ {page}페이지 완료")

driver.quit()

# CSV 저장
df = pd.DataFrame(data)
df.to_csv("bizinfo_지원사업_공고목록.csv", index=False, encoding="utf-8-sig")
print(f"🎉 수집 완료: 총 {len(df)}건 저장")

▶ 총 수집 페이지 수: 17
✅ 1페이지 완료
✅ 2페이지 완료
✅ 3페이지 완료
✅ 4페이지 완료
✅ 5페이지 완료
✅ 6페이지 완료
✅ 7페이지 완료
✅ 8페이지 완료
✅ 9페이지 완료
✅ 10페이지 완료
✅ 11페이지 완료
✅ 12페이지 완료
✅ 13페이지 완료
✅ 14페이지 완료
✅ 15페이지 완료
✅ 16페이지 완료
✅ 17페이지 완료
🎉 수집 완료: 총 251건 저장


## 📁 공고에서 첨부파일 다운로드 받기

In [10]:
# 🔽 CSV 파일 경로 지정 (수정 필요)
csv_path = "bizinfo_지원사업_공고목록.csv"

# CSV 불러오기
df = pd.read_csv(csv_path)

def extract_pblanc_id(link):
    if isinstance(link, str) and "pblancId=" in link:
        parsed = urlparse(link)
        query = parse_qs(parsed.query)
        return query.get("pblancId", [None])[0]
    return None

df["pblanc_id"] = df["링크"].apply(extract_pblanc_id)
df = df.dropna(subset=["pblanc_id"])
df.to_csv("bizinfo_지원사업_공고목록.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ✅ ZIP 깨진 한글 파일명 복구 & 압축 해제 함수
def extract_zip_with_encoding(zip_path, extract_to, encoding="euc-kr"):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for zip_info in zip_ref.infolist():
            try:
                decoded_name = zip_info.filename.encode('cp437').decode(encoding)
                zip_info.filename = decoded_name  # 복구된 이름으로 설정
                zip_ref.extract(zip_info, path=extract_to)
                print(f"📄 추출됨: {decoded_name}")
            except Exception as e:
                print(f"❌ 파일 추출 실패: {zip_info.filename} ({e})")

In [ ]:
# ✅ 첨부파일 다운로드 함수
def download_files(pblanc_id):
    save_dir = os.path.join(os.getcwd(), "download(support)", pblanc_id)
    os.makedirs(save_dir, exist_ok=True)

    url = f"https://www.bizinfo.go.kr/web/lay1/bbs/S1T122C128/AS/74/view.do?pblancId={pblanc_id}"

    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        links = soup.select("a.icon_download")
        if not links:
            print(f"❌ 첨부파일 없음: {pblanc_id}")
            return

        for link in links:
            file_name = link["title"].replace("첨부파일 ", "").replace(" 다운로드", "").strip()
            file_url = "https://www.bizinfo.go.kr" + link["href"]

            try:
                file_res = requests.get(file_url)
                file_res.raise_for_status()

                file_path = os.path.join(save_dir, file_name)
                with open(file_path, "wb") as f:
                    f.write(file_res.content)

                # ✅ ZIP 처리
                if file_name.lower().endswith(".zip"):
                    try:
                        extract_zip_with_encoding(file_path, save_dir, encoding="euc-kr")
                        os.remove(file_path)
                        print(f"✅ ZIP 압축 해제 및 삭제: {file_path}")
                    except zipfile.BadZipFile:
                        print(f"⚠️ 손상된 ZIP 파일: {file_path}")
                else:
                    print(f"✅ 저장 완료: {file_path}")

            except Exception as e:
                print(f"❌ 다운로드 실패: {file_name} ({e})")

    except Exception as e:
        print(f"❌ 요청 실패: {pblanc_id} ({e})")

In [ ]:
# df = df.head(20)

# 모든 공고 ID에 대해 파일 다운로드
for pblanc_id in df["pblanc_id"].unique():
    download_files(pblanc_id)
    print("✨ 전체 다운로드 완료")

## ✏️ 다운된 HWP 파일들 모두 변환해서 JSON 파일로 다시 저장

In [ ]:
import os
import olefile
import shutil
import json
import uuid

In [ ]:
# 원래 저장되어 있던 경로로 다시 저장되게
base_dir = os.path.join(os.getcwd(), "download(support)")
for root, dirs, files in os.walk(base_dir):
    print("현재 폴더:", root)
output_json_dir = root 
os.makedirs(output_json_dir, exist_ok=True)

def extract_text_from_hwp(file_path):
    try:
        temp_path = os.path.join(os.getcwd(), "temp.hwp")
        shutil.copy(file_path, temp_path)

        with olefile.OleFileIO(temp_path) as f:
            encoded_text = f.openstream('PrvText').read()
            decoded_text = encoded_text.decode('utf-16')

        os.remove(temp_path)  # 이제 안전하게 제거 가능
        return decoded_text

    except Exception as e:
        print(f"❌ 변환 실패: {file_path} - {e}")
        return None

def convert_all_hwp_to_json(base_dir):
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.lower().endswith(".hwp"):
                full_path = os.path.join(root, file)
                text = extract_text_from_hwp(full_path)
                if text:
                    # 파일 이름 유지, 확장자만 .json으로 변경
                    json_name = os.path.splitext(file)[0] + ".json"
                    json_path = os.path.join(output_json_dir, json_name)

                    # json 파일로 저장
                    with open(json_path, "w", encoding="utf-8") as f:
                        json.dump({"file_name": file, "text": text}, f, ensure_ascii=False, indent=2)

                    print(f"✅ 변환 완료: {file} → {json_name}")

convert_all_hwp_to_json(base_dir)


현재 폴더: c:\Users\HY\Desktop\KB AI Challenge\download(support)
현재 폴더: c:\Users\HY\Desktop\KB AI Challenge\download(support)\PBLN_000000000112804
현재 폴더: c:\Users\HY\Desktop\KB AI Challenge\download(support)\PBLN_000000000112806
✅ 변환 완료: 2025+하반기+소상공인시장진흥자금+신청서+등(부동산+담보용).hwp → 2025+하반기+소상공인시장진흥자금+신청서+등(부동산+담보용).json
✅ 변환 완료: 2025+하반기+소상공인진흥자금+공고문.hwp → 2025+하반기+소상공인진흥자금+공고문.json


## 📦 벡터스토어 저장

In [ ]:
# 제외 키워드 : '지원서', '사업계획서', '계획서', '서식', '양식', '신청서', '동의서', '서약서', '신청서', '의향서', '확인서'
# 저장된 공고는 csv 파일의 '벡터스토어 저장'부분에 TRUE 해놓기
# 벡터스토어에 저장할 때 메타데이터에 공고명이랑 pblanc_id도 저장해놓자...

In [ ]:
# 제외할 키워드 목록
EXCLUDE_KEYWORDS = ['지원서', '사업계획서', '계획서', '서식', '양식', '신청서', 
                    '동의서', '서약서', '의향서', '확인서']

# 마침표 뒤가 아닌 줄바꿈 제거 함수
def remove_newlines_except_after_period(text):
    return re.sub(r'(?<!\.)(\n|\r\n)', ' ', text)

# 파일 로딩 함수 (PDF, JSON)
def load_document(file_path, folder_name, title):
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf":
        loader = PDFPlumberLoader(file_path)
    elif ext == ".json":
        loader = TextLoader(file_path, encoding="utf-8")
    else:
        return []

    docs = loader.load()
    for doc in docs:
        doc.page_content = remove_newlines_except_after_period(doc.page_content)
        doc.metadata["folder_name"] = folder_name
        doc.metadata["file_type"] = ext
        doc.metadata["title"] = title  # ✅ 공고명
    return docs


# 전체 벡터스토어 생성 및 저장 함수
def save_vectorstore_from_folders(base_dir, vectorstore_base_dir, csv_path, openai_api_key):
    df = pd.read_csv(csv_path)
    if "벡터스토어 저장" not in df.columns:
        df["벡터스토어 저장"] = False

    for folder_name in os.listdir(base_dir):
        folder_path = os.path.join(base_dir, folder_name)
        if not os.path.isdir(folder_path):
            continue

        # 🔍 공고명(title) 가져오기
        match_row = df[df["pblanc_id"].astype(str) == folder_name]
        if match_row.empty:
            print(f"❌ 공고명 찾을 수 없음: {folder_name}")
            continue
        title = match_row.iloc[0]["제목"]

        all_docs = []
        for file_name in os.listdir(folder_path):
            if not (file_name.endswith(".pdf") or file_name.endswith(".json")):
                continue
            if any(keyword in file_name for keyword in EXCLUDE_KEYWORDS):
                continue

            file_path = os.path.join(folder_path, file_name)
            docs = load_document(file_path, folder_name, title)
            all_docs.extend(docs)

        if all_docs:
            print(f"📄 {folder_name}: {len(all_docs)}개 문서 처리 중...")
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
            split_docs = text_splitter.split_documents(all_docs)
            embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)
            vectorstore = FAISS.from_documents(split_docs, embeddings)

            folder_vector_path = os.path.join(vectorstore_base_dir, folder_name)
            os.makedirs(folder_vector_path, exist_ok=True)
            vectorstore.save_local(folder_vector_path)

            with open(os.path.join(folder_vector_path, "documents.pkl"), "wb") as f:
                pickle.dump(split_docs, f)

            df.loc[df["pblanc_id"].astype(str) == folder_name, "벡터스토어 저장"] = True

    df.to_csv(csv_path, index=False)
    print("✅ 모든 벡터스토어 저장 및 CSV 업데이트 완료.")

In [30]:
base_dir = os.path.join(os.getcwd(), "download(support)")
vectorstore_base_dir = os.path.join(os.getcwd(), "embeddings")
csv_path = os.path.join(os.getcwd(), "bizinfo_지원사업_공고목록.csv")

save_vectorstore_from_folders(base_dir, vectorstore_base_dir, csv_path, openai_api_key)

📄 PBLN_000000000112806: 1개 문서 처리 중...
✅ 모든 벡터스토어 저장 및 CSV 업데이트 완료.
